# Cloud Provider Analytics — pipeline de la segunda entrega

Grupo 13 2026-2Q - 72.80 Big Data (ITBA). Corre el pipeline completo
Landing → Bronze → Silver → Gold → Cassandra/AstraDB con el código python del repositorio

## 1. Repositorio y dependencias

In [ ]:
import os, sys, pathlib
REPO_URL = "https://github.com/agustinmorantes/tp-big-data.git"
BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules
ROOT = pathlib.Path("/content/tp-big-data") if IN_COLAB else pathlib.Path.cwd().parent

if IN_COLAB and not ROOT.exists():
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {ROOT}
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ["PYTHONPATH"] = str(ROOT / "src")
print("repo:", ROOT)

In [ ]:
# Colab trae Java 11+; PySpark 3.5.1 es la versión con la que se probó el pipeline.
%pip install -q pyspark==3.5.1 cassandra-driver==3.29.3

## 2. Dataset

Subir el .zip del dataset de la cátedra.

In [ ]:
import zipfile, shutil
landing = ROOT / "data" / "datalake" / "landing"
if not (landing / "usage_events_stream").exists():
    if IN_COLAB:
        from google.colab import files
        uploaded = files.upload()          # elegir el .zip del dataset
        zip_name = next(iter(uploaded))
    else:
        zip_name = input("Ruta al .zip del dataset: ")
    tmp = ROOT / "data" / "_unzip"
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(tmp)
    src = next(tmp.rglob("landing"))
    landing.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(src), landing)
    shutil.rmtree(tmp, ignore_errors=True)
print(sorted(p.name for p in landing.iterdir()))
print("JSONL:", len(list((landing / "usage_events_stream").glob("*.jsonl"))))

## 3. AstraDB

Antes de correr:

1. En la consola de Astra, crear (o reusar) una base Serverless (Cassandra) y, en Keyspaces, agregar el keyspace `cloud_provider_analytics`. En Astra el keyspace no se puede crear por CQL.
2. Generar un Application Token con rol Database Administrator (crea tablas) y descargar el Secure Connect Bundle.

In [ ]:
from getpass import getpass
TARGET = "astra"   # "astra" o "cassandra"
os.environ["CPA_SERVING_TARGET"] = TARGET
if TARGET == "astra":
    bundle = ROOT / "config" / "secure-connect.zip"     # ignorado por git (config/secure-connect*.zip)
    if not bundle.exists():
        if IN_COLAB:
            from google.colab import files
            up = files.upload()                           # elegir secure-connect-<db>.zip
            pathlib.Path(next(iter(up))).rename(bundle)
        else:
            shutil.copy(input("Ruta al secure connect bundle: "), bundle)
    os.environ["ASTRA_SECURE_CONNECT_BUNDLE"] = str(bundle)
    os.environ.setdefault("ASTRA_DB_KEYSPACE", "cloud_provider_analytics")
    if not os.environ.get("ASTRA_DB_APPLICATION_TOKEN"):
        os.environ["ASTRA_DB_APPLICATION_TOKEN"] = getpass("Astra token (AstraCS:...): ")

from cpa.config import load_config
from cpa.serving import connect
settings = load_config().serving
cluster, session = connect(settings)
print("conectado a", settings["target"], "-", session.execute("SELECT release_version FROM system.local").one()[0])
cluster.shutdown()

## 4. Pipeline

Tarda ~10–15 min en Colab.

In [ ]:
!cd {ROOT} && python -m cpa.run_pipeline reset all speed-view idempotency evidence 2>&1 | grep -E "cpa\.|Error|Traceback" | cut -c1-260

## 5. Evidencia

In [ ]:
import json
from datetime import date
import pandas as pd
import pyarrow.dataset as ds
from cpa.common import dir_stats

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)
lake = ROOT / "data" / "datalake"
ev = ROOT / "evidence" / "segunda-entrega"
metrics = [json.loads(l) for l in (lake / "_meta" / "run_metrics.jsonl").read_text(encoding="utf-8").splitlines()]

def tablas(zona):
    return sorted(p for p in (lake / zona).iterdir() if p.is_dir() and not p.name.startswith(("_", ".")))

def leer(path):
    return ds.dataset(path, format="parquet", partitioning="hive").to_table().to_pandas()

# Inventario del lake: filas, archivos Parquet y tamaño por tabla
filas = []
for zona in ["bronze", "silver", "gold", "quarantine"]:
    for t in tablas(zona):
        st = dir_stats(t)
        filas.append({"tabla": f"{zona}/{t.name}",
                      "filas": ds.dataset(t, format="parquet", partitioning="hive").count_rows(),
                      "archivos": st["parquet_files"], "kb": round(st["bytes"] / 1024, 1)})
print(pd.DataFrame(filas).to_string(index=False))

In [ ]:
# Streaming: un renglón por micro-batch de Silver. Las corridas que siguen a la primera son las de
# la prueba de idempotencia (re-ejecución sin input nuevo y reproceso sin checkpoint).
batches = pd.DataFrame([m for m in metrics if m["stage"] == "silver_events_batch"])
batches["corrida"] = (batches.batch_id == 0).cumsum()
print(batches[["corrida", "batch_id", "input", "silver_new", "rejected", "duplicates", "remediated",
               "late_events", "negative_cost_flagged"]].to_string(index=False))

speed = [m for m in metrics if m["stage"] == "speed_view"][-1]
print(f"\nVista con watermark nativo de {speed['watermark_delay']}: {speed['input_rows']} filas de entrada, "
      f"{speed['rows_dropped_by_watermark']} descartadas ({speed['dropped_pct']} %)")

In [ ]:
# Idempotencia: conteos antes, después de re-ejecutar y después de reprocesar sin checkpoint
filas = []
for linea in (ev / "idempotencia.md").read_text(encoding="utf-8").splitlines():
    c = [x.strip(" `") for x in linea.strip().strip("|").split("|")]
    if len(c) == 6 and c[0] in ("lake", "cassandra"):
        filas.append(c[:5])
idem = pd.DataFrame(filas, columns=["destino", "tabla", "antes", "reejecucion", "reproceso"])
idem["igual"] = ((idem.antes == idem.reejecucion) & (idem.reejecucion == idem.reproceso)).map({True: "si", False: "no"})
print(idem.to_string(index=False))
print("\nquarantine/usage_events crece en el reproceso porque guarda los eventos releídos que se descartaron como duplicados.")

In [ ]:
# Anomalías de costo por MAD (|z robusto| > 3.5 con al menos 14 días de historia)
a = leer(lake / "gold" / "cost_anomaly_mart")
anom = a[a.is_anomaly]
print(f"días (org, servicio) puntuados: {len(a)}")
print(f"anomalías: {len(anom)} ({100 * len(anom) / len(a):.2f} %)")
print(anom.anomaly_type.value_counts().to_string())
print()
cols = ["org_id", "usage_date", "service", "daily_cost_usd", "median_cost_usd", "robust_z"]
print(anom.sort_values("robust_z", ascending=False)[cols].head(10).round(2).to_string(index=False))

### Consultas sobre Cassandra/AstraDB

In [ ]:
# Se ejecutan las mismas consultas que la etapa `queries`, con los parámetros de esa corrida
from cpa.queries import build_queries
from cpa.serving import connect

p = [m for m in metrics if m["stage"] == "queries"][-1]["params"]
consultas = build_queries({"org_id": p["org_id"],
                           "last_usage_date": date.fromisoformat(p["last_usage_date"]),
                           "last_ticket_date": date.fromisoformat(p["last_ticket_date"])})
cluster, session = connect(settings)
session.set_keyspace(settings["keyspace"])
print("destino:", settings["target"], "\n")
for q in consultas:
    rs = session.execute(q["cql"], q["params"])
    df = pd.DataFrame(list(rs), columns=rs.column_names)
    print(f"{q['id'].upper()}. {q['title']} ({len(df)} filas)")
    print(df.head(10).round(4).to_string(index=False), "\n")
cluster.shutdown()

### Quarantine (conteos por motivo y muestras)

In [ ]:
q = pd.concat([leer(t) for t in tablas("quarantine")], ignore_index=True)
conteo = q.groupby(["source_table", "disposition", "primary_reason"]).size().rename("filas").reset_index()
print(conteo.to_string(index=False))
print()
for _, r in q.drop_duplicates("primary_reason").iterrows():
    print(f"{r.source_table} / {r.primary_reason}:\n  {r.record_json[:220]}")

### Tests

13 pruebas de reglas de calidad, marts Gold y streaming de punta a punta sobre fixtures chicos.

In [ ]:
%pip install -q pytest==8.3.3
!cd {ROOT} && python -m pytest -q -p no:cacheprovider